# 05c — Port fidele de `Dietetique.Load` : fusion LINQ a l'echelle reelle

> **Slice A du capstone MealPlanner** ([issue #4617](https://github.com/jsboige/CoursIA/issues/4617), `See #1206`).
> Source distillee : `MyIntelligenceAgency/Z3.LinqBinding@EPFdevelopment`, projet `Demo2/MealPlanning`
> (`PlanificateurDeMenus.cs`, `Ciqual.cs`, `Recettes.cs`).

Les notebooks [05d](05d_Hierarchical_Patient_Restrictions.ipynb) (axe **structurel**, 138 plats) et
[05e](05e_Meal_Planner_Convergence_Scale.ipynb) (axe **nutritionnel**, jointure lexicale anglaise) sont
de bonnes mises a l'echelle, mais aucun ne **porte fidelement** la couche de donnees du Demo2 d'origine :
la methode `Dietetique.Load`. Ce notebook comble exactement ce trou.

## Ce que `Dietetique.Load` fait reellement

C'est un pipeline **LINQ-to-objects** qui fusionne deux sources heterogenes en un catalogue
nutritionnel exploitable par le solveur :

1. **Ciqual ANSES** (table officielle de composition nutritionnelle, 4 fichiers XML) — ici la table
   **2025** (DOI 10.57745/RDMHWY, licence etalab 2.0), de schema identique a la 2017 du Demo2.
2. **Recettes OpenData** (denrees / plats / menus de cantine, JSON).

Le pipeline enchaine :

- **Deux jointures** `CONST -> COMPO -> ALIM` pour reconstituer, par constituant, la teneur de chaque aliment ;
- un **matcheur lexical flou** maison (`CalculeProximiteLexicale`, sac-de-mots) pour rapprocher les
  denrees des recettes (libelles bruts de cantine, `POIVRE GRIS MOULU K`) des aliments Ciqual (`Poivre noir`) ;
- une **agregation** `Zip`/`Aggregate` pour composer la teneur d'un plat a partir de celles de ses denrees ;
- la serialisation d'un cache `dietetique.json`.

**Pourquoi c'est un bon probleme (Prong B)** : ce n'est pas un `join` jouet a 7 lignes. Les denrees de
cantine n'ont **aucune cle commune** avec Ciqual — seul un appariement **textuel approximatif** les relie,
sur 3484 aliments x ~200 denrees. C'est de la **fusion de donnees reelle**, le genre que LINQ absorbe
declarativement la ou un script imperatif deviendrait illisible.

**Perimetre** : Slice A s'arrete **avant Z3** (le theoreme hierarchique + la convergence a l'echelle sont
les Slices C/D du capstone). Son livrable est la couche `Dietetique` fidele que le solveur consommera.


## 0. Dependances et donnees

Le corpus (Ciqual 69 Mo + Recettes) est **trop volumineux pour le depot** : il est `.gitignore` sous
`data/meals/`. Recuperez-le une fois via le telechargeur livre avec le capstone :

```bash
python download_meal_data.py          # Ciqual 2025 + Recettes OpenData + RecipeML
```

La cellule suivante resout le chemin des donnees et **echoue franchement** (fail-loud) si elles sont
absentes, en rappelant la commande — pas de jeu de substitution.


In [1]:
#r "nuget: Newtonsoft.Json, 13.0.3"
using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Text;
using System.Text.RegularExpressions;
using System.Xml.Serialization;
using Newtonsoft.Json;

// Resolution robuste du dossier data/meals (relatif au notebook, avec repli remontant).
string ResolveDataDir()
{
    var candidates = new[]
    {
        Path.Combine(Directory.GetCurrentDirectory(), "data", "meals"),
        Path.Combine(Directory.GetCurrentDirectory(), "MyIA.AI.Notebooks", "SymbolicAI", "SMT", "Z3", "data", "meals"),
    };
    foreach (var c in candidates)
        if (Directory.Exists(Path.Combine(c, "Ciqual")) && Directory.Exists(Path.Combine(c, "Recettes")))
            return c;
    return candidates[0];
}

var DataDir = ResolveDataDir();
var ciqualDir = Path.Combine(DataDir, "Ciqual");
var recettesDir = Path.Combine(DataDir, "Recettes");

// Affichage relatif au repertoire courant (evite de fuiter un chemin machine absolu).
string Rel(string p) => Path.GetRelativePath(Directory.GetCurrentDirectory(), p);

if (!Directory.Exists(ciqualDir) || !Directory.Exists(recettesDir))
{
    Console.WriteLine("[DONNEES ABSENTES] " + Rel(DataDir));
    Console.WriteLine("Lancez d'abord :  python download_meal_data.py");
    Console.WriteLine("(Ciqual ANSES 2025 + Recettes OpenData ; corpus .gitignore, ~70 Mo)");
}
else
{
    Console.WriteLine("Donnees OK : " + Rel(DataDir));
    Console.WriteLine("  Ciqual  : " + Directory.GetFiles(ciqualDir, "*.xml").Length + " fichiers XML");
    Console.WriteLine("  Recettes: " + Directory.GetFiles(recettesDir, "*.json").Length + " fichiers JSON");
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Newtonsoft.Json, 13.0.3

Donnees OK : data\meals


  Ciqual  : 4 fichiers XML


  Recettes: 6 fichiers JSON


## 1. Charger Ciqual (ANSES 2025) via `XmlSerializer`

Le Demo2 deserialise 4 tables XML en classes POCO. Pour Slice A, seules trois portent la fusion :

| Table | Cle(s) | Champs utiles |
|-------|--------|---------------|
| `CONST` | `const_code` | `const_nom_fr` (nom du constituant : energie, proteines, ...) |
| `COMPO` | `alim_code` + `const_code` | `teneur` (valeur pour 100 g) |
| `ALIM`  | `alim_code` | `alim_nom_fr` (nom de l'aliment) |

`COMPO` est le **gros fichier** (~258 000 lignes : chaque couple aliment x constituant). On garde la
deserialisation par `XmlSerializer` du Demo2 (fidelite), seuls les noms de fichiers passent en 2025.


In [2]:
using System.Xml.Linq;

// --- POCOs Ciqual (memes champs que Ciqual.cs, peuples par LINQ-to-XML) ---
// Note : XmlSerializer ne fonctionne pas sur des types definis dans une cellule .NET Interactive
// (l'assembly de serialisation generee bute sur les noms 'Submission#N', non CLS-compliant).
// On parse donc avec XDocument (LINQ-to-XML) : le pipeline de fusion en aval est inchange.
public class CConst { public uint const_code; public string const_nom_fr; public string const_nom_eng; }
public class CAlim  { public uint alim_code;  public string alim_nom_fr;  public string alim_nom_eng;  }
public class CCompo { public uint alim_code;  public uint const_code;      public string teneur;        }

public class TableConstituant { public CConst[] CONST { get; set; } }
public class TableAliments    { public CAlim[]  ALIM  { get; set; } }
public class TableComposition { public CCompo[] COMPO { get; set; } }

public class Ciqual
{
    public TableConstituant Constituants { get; set; }
    public TableAliments Aliments { get; set; }
    public TableComposition Compositions { get; set; }

    static string Txt(XElement e, string n) => (string)e.Element(n);          // texte brut (espaces preserves, comme la source)
    static uint   U  (XElement e, string n) => uint.Parse(((string)e.Element(n)).Trim());

    public static Ciqual Load(string folderPath)
    {
        var constDoc = XDocument.Load(Path.Combine(folderPath, "const_2025_11_03.xml"));
        var alimDoc  = XDocument.Load(Path.Combine(folderPath, "alim_2025_11_03.xml"));
        var compoDoc = XDocument.Load(Path.Combine(folderPath, "compo_2025_11_03.xml")); // ~69 Mo
        return new Ciqual
        {
            Constituants = new TableConstituant { CONST = constDoc.Root.Elements("CONST")
                .Select(e => new CConst { const_code = U(e, "const_code"), const_nom_fr = Txt(e, "const_nom_fr"), const_nom_eng = Txt(e, "const_nom_eng") }).ToArray() },
            Aliments = new TableAliments { ALIM = alimDoc.Root.Elements("ALIM")
                .Select(e => new CAlim { alim_code = U(e, "alim_code"), alim_nom_fr = Txt(e, "alim_nom_fr"), alim_nom_eng = Txt(e, "alim_nom_eng") }).ToArray() },
            Compositions = new TableComposition { COMPO = compoDoc.Root.Elements("COMPO")
                .Select(e => new CCompo { alim_code = U(e, "alim_code"), const_code = U(e, "const_code"), teneur = Txt(e, "teneur") }).ToArray() },
        };
    }
}

var ciqual = Ciqual.Load(ciqualDir);
Console.WriteLine($"Constituants (CONST) : {ciqual.Constituants.CONST.Length}");
Console.WriteLine($"Aliments    (ALIM)   : {ciqual.Aliments.ALIM.Length}");
Console.WriteLine($"Compositions (COMPO) : {ciqual.Compositions.COMPO.Length}");
Console.WriteLine();
Console.WriteLine("Exemple d'aliment : " + ciqual.Aliments.ALIM[0].alim_nom_fr
    + " (code " + ciqual.Aliments.ALIM[0].alim_code + ")");


Constituants (CONST) : 74


Aliments    (ALIM)   : 3484


Compositions (COMPO) : 257816


Exemple d'aliment :  Pastis  (code 1000)


## 2. Charger les Recettes (OpenData cantine) via JSON

Trois fichiers : `denrees*` (ingredients bruts), `plats*` (recettes), `menus*` (qui ordonne les plats
par creneau `ordre_plat`). On reproduit `Recettes.Load` : `Newtonsoft.Json`, et le repli `libelle_denree`
<- `libelle_recette` quand le libelle de denree est vide (exactement comme la source).


In [3]:
// --- POCOs Recettes (fideles a Recettes.cs) ---
public class DenreeRecord { public Fields fields { get; set; }
    public class Fields { public string libelle_recette { get; set; } public string code_recette { get; set; } public string libelle_denree { get; set; } } }
public class PlatRecord { public Fields fields { get; set; }
    public class Fields { public string code_recette { get; set; } public string code_plat { get; set; } public string libelle_plat { get; set; } } }
public class MenuRecord { public Fields fields { get; set; }
    public class Fields { public string code_plat { get; set; } public int ordre_plat { get; set; } public string libelle_plat { get; set; } } }

public class Recettes
{
    public List<DenreeRecord> Denrees { get; set; } = new();
    public List<PlatRecord> Plats { get; set; } = new();
    public List<MenuRecord> Menus { get; set; } = new();

    public static Recettes Load(string folderPath)
    {
        var r = new Recettes();
        var ser = new JsonSerializer();
        List<T> ReadAll<T>(string pattern)
        {
            var acc = new List<T>();
            foreach (var f in Directory.GetFiles(folderPath, pattern))
            {
                using var reader = File.OpenText(f);
                using var jr = new JsonTextReader(reader);
                acc.AddRange(ser.Deserialize<List<T>>(jr));
            }
            return acc;
        }
        r.Denrees = ReadAll<DenreeRecord>("denrees*");
        foreach (var d in r.Denrees)
            if (string.IsNullOrEmpty(d.fields.libelle_denree)) d.fields.libelle_denree = d.fields.libelle_recette;
        r.Plats = ReadAll<PlatRecord>("plats*");
        r.Menus = ReadAll<MenuRecord>("menus*");
        return r;
    }
}

var recettes = Recettes.Load(recettesDir);
Console.WriteLine($"Denrees : {recettes.Denrees.Count}");
Console.WriteLine($"Plats   : {recettes.Plats.Count}");
Console.WriteLine($"Menus   : {recettes.Menus.Count}");
Console.WriteLine();
Console.WriteLine("Exemple de denree (libelle brut de cantine) : " + recettes.Denrees[0].fields.libelle_denree);


Denrees : 698


Plats   : 493


Menus   : 1089


Exemple de denree (libelle brut de cantine) : POIVRE GRIS MOULU K


## 3. Le matcheur lexical flou `CalculeProximiteLexicale`

Le coeur du port. Les denrees de cantine (`POIVRE GRIS MOULU K`, `EMMENTHAL RAPE/KG+`) n'ont **aucune cle**
vers Ciqual (`Poivre noir`, `Emmental`). Le Demo2 les rapproche par un score **sac-de-mots** :

1. `CreateSimpleList` normalise chaque chaine : majuscules, suppression des non-alphanumeriques, puis
   **troncature des suffixes** `S` puis `E` (pluriels / feminins grossiers), mots de longueur > 1 conserves.
2. `CalculeProximiteLexicale` somme, pour chaque mot de `s2` retrouve dans `s1`, un poids
   `1 / (2*(idx + position) + 1)` qui **decroit** avec la position du mot — un mot commun en tete pese plus
   qu'un mot commun en fin. Un petit malus `-(|bag1| + |bag2|)/1000` departage les chaines courtes.

C'est volontairement simple (pas de Levenshtein, pas de TF-IDF) : l'interet pedagogique est de voir un
**heuristique d'appariement** se brancher dans un pipeline LINQ declaratif.


In [4]:
static List<string> CreateSimpleList(string s)
{
    var toReturn = new List<string>();
    foreach (var mot in s.Split(new[] { ' ' }, StringSplitOptions.RemoveEmptyEntries))
    {
        var m = Regex.Replace(mot.ToUpperInvariant(), @"\W", "");
        m = m.TrimEnd('S').TrimEnd('E');
        if (m.Length > 1) toReturn.Add(m);
    }
    return toReturn;
}

static double CalculeProximiteLexicale(string s1, string s2)
{
    var bag1 = CreateSimpleList(s1);
    var bag2 = CreateSimpleList(s2);
    var toReturn = 0d;
    for (var wi = 0; wi < bag2.Count; wi++)
    {
        var idx = bag1.IndexOf(bag2[wi]);
        if (idx > -1) toReturn += 1.0 / (2.0 * (idx + wi) + 1.0);
    }
    toReturn -= (bag1.Count + bag2.Count) / 1000.0;
    return toReturn;
}

// Demonstration : on score quelques paires (denree de cantine -> aliment Ciqual candidat).
void Demo(string denree, string aliment)
    => Console.WriteLine($"  prox({aliment,-26} , {denree,-24}) = {CalculeProximiteLexicale(aliment, denree):0.0000}");
Demo("SUCRE SEMOULE 1KG", "Sucre blanc");
Demo("SUCRE SEMOULE 1KG", "Boeuf braise");
Demo("TOMATES CONCASSEES 5/1", "Tomate, pulpe");
Demo("EMMENTHAL RAPE/KG+", "Emmental");


  prox(Sucre blanc                , SUCRE SEMOULE 1KG       ) = 0,9950


  prox(Boeuf braise               , SUCRE SEMOULE 1KG       ) = -0,0050


  prox(Tomate, pulpe              , TOMATES CONCASSEES 5/1  ) = 0,9950


  prox(Emmental                   , EMMENTHAL RAPE/KG+      ) = -0,0030


## 4. `Dietetique.Load` : les deux jointures + le matching + l'agregation `Zip`

On assemble maintenant le pipeline complet, **fidele a la source**. Les etapes, dans l'ordre :

1. **Constituants** : projection de `CONST` sur son seul `const_nom_fr`.
2. **`constituantsAliments`** : pour **chaque** constituant, deux jointures `CONST ⋈ COMPO ⋈ ALIM`
   donnent le tableau des `(NomAliment, Teneur)` de teneur > 0. La teneur est nettoyee comme la source
   (`traces`->0, `<`->espace, `-`->0, culture `fr` pour la virgule decimale).
3. **`matchingMots`** : chaque denree distincte est appariee a l'aliment de **proximite lexicale maximale**.
4. **`Denrees`** : chaque denree matchee recoit le vecteur de teneurs (une valeur par constituant).
5. **`Plats`** : `Plats ⋈ Menus` (pour l'`ordre_plat`), puis `GroupJoin` avec les denrees du plat ; la
   composition d'un plat est l'**agregation `Zip`** (somme terme a terme) des compositions de ses denrees.

> Le pipeline tourne en quelques dizaines de secondes (74 constituants x 258 000 compositions + ~200 x 3484
> appariements lexicaux). C'est le cout reel d'une fusion sur donnees brutes — borne et raisonnable.


In [5]:
// --- Types du catalogue (fideles a PlanificateurDeMenus.cs) ---
public class Constituant { public string Nom { get; set; } }
public class DenreeImport { public string Nom { get; set; } = ""; public decimal[] Compositions { get; set; } }
public class Ingredient { public int Denree { get; set; } public decimal Quantite { get; set; } }
public class PlatImport { public string Nom { get; set; } public int Ordre { get; set; }
    public List<Ingredient> Ingredients { get; set; } = new(); public decimal[] Compositions { get; set; } }

public class Dietetique
{
    public List<Constituant> Constituants { get; set; }
    public List<DenreeImport> Denrees { get; set; }
    public List<PlatImport> Plats { get; set; }

    public static Dietetique Load(Ciqual ciqual, Recettes recettes)
    {
        var toReturn = new Dietetique();

        // 1. Liste des constituants (nom seul).
        toReturn.Constituants = ciqual.Constituants.CONST
            .Select(c => new Constituant { Nom = c.const_nom_fr }).ToList();

        // 2. Deux jointures : par constituant, le tableau des (aliment, teneur>0).
        var fr = new CultureInfo("fr");
        var constituantsAliments = toReturn.Constituants
            .Select(c => ciqual.Constituants.CONST
                .Where(cc => cc.const_nom_fr == c.Nom).ToArray()
                .Join(ciqual.Compositions.COMPO, cc => cc.const_code, compo => compo.const_code,
                    (cc, compo) => new
                    {
                        Teneur = decimal.Parse(compo.teneur.Replace("traces", "0")
                            .Replace('<', ' ').Replace('-', '0').Trim(), fr),
                        CodeAlim = compo.alim_code
                    }).ToArray()
                .Join(ciqual.Aliments.ALIM, compo => compo.CodeAlim, alim => alim.alim_code,
                    (compo, alim) => new { NomAliment = alim.alim_nom_fr, compo.Teneur })
                .Where(ca => ca.Teneur > 0)
                .ToArray())
            .ToArray();

        // 3. Denrees distinctes non vides, appariees a l'aliment de proximite lexicale maximale.
        var denreesFiltered = recettes.Denrees
            .Where(d => !string.IsNullOrEmpty(d.fields.libelle_denree))
            .GroupBy(d => d.fields.libelle_denree).Select(g => g.First()).ToArray();

        var matchingMots = denreesFiltered
            .Select(d => new
            {
                NomDenree = d.fields.libelle_denree,
                NomAliment = constituantsAliments[0]
                    .OrderByDescending(a => CalculeProximiteLexicale(a.NomAliment, d.fields.libelle_denree))
                    .First().NomAliment,
                Proximite = constituantsAliments[0]
                    .Max(a => CalculeProximiteLexicale(a.NomAliment, d.fields.libelle_denree))
            })
            .Where(m => m.NomAliment != ciqual.Aliments.ALIM[0].alim_nom_fr)
            .ToDictionary(a => a.NomDenree, a => new { Nom = a.NomAliment, a.Proximite });

        // 4. Vecteur de teneurs par denree matchee (une valeur par constituant).
        toReturn.Denrees = matchingMots
            .Select(d => new DenreeImport
            {
                Nom = d.Key,
                Compositions = constituantsAliments
                    .Select(ca => ca.FirstOrDefault(a => a.NomAliment == matchingMots[d.Key].Nom)?.Teneur ?? 0)
                    .ToArray()
            }).ToList();

        // 5. Plats : jointure avec Menus (ordre), GroupJoin avec les denrees, composition par agregation Zip.
        var denreesFilteredAvecDoublons = recettes.Denrees
            .Where(d => denreesFiltered.Any(df => df.fields.libelle_denree == d.fields.libelle_denree)).ToArray();

        var denreesPlats = recettes.Plats
            .Join(recettes.Menus, p => p.fields.code_plat, m => m.fields.code_plat,
                (p, m) => new { Plat = p, Ordre = m.fields.ordre_plat })
            .GroupBy(p => p.Plat.fields.libelle_plat).Select(g => g.First())
            .GroupJoin(denreesFilteredAvecDoublons, p => p.Plat.fields.code_recette, d => d.fields.code_recette,
                (plat, dp) => new { NomPlat = plat.Plat.fields.libelle_plat, Ordre = plat.Ordre, DenreesPlat = dp.ToArray() })
            .Where(p => p.DenreesPlat.Length > 0)
            .ToArray();

        toReturn.Plats = denreesPlats.Select(pd => new PlatImport
        {
            Nom = pd.NomPlat,
            Ordre = pd.Ordre,
            Ingredients = pd.DenreesPlat.Select(dp => new Ingredient
            {
                Denree = toReturn.Denrees.FindIndex(d => d.Nom == dp.fields.libelle_denree),
                Quantite = 1
            }).Where(i => i.Denree >= 0).ToList(),
            Compositions = toReturn.Denrees
                .Where(denree => pd.DenreesPlat.Any(dp => dp.fields.libelle_denree == denree.Nom))
                .Select(denree => denree.Compositions)
                .Aggregate(new decimal[toReturn.Constituants.Count],
                    (c1, c2) => c1.Zip(c2, (a, b) => a + b).ToArray())
        }).ToList();

        return toReturn;
    }
}

Console.WriteLine("Dietetique.Load defini.");


Dietetique.Load defini.


## 5. Executer la fusion, explorer le resultat, regenerer le cache

On lance le pipeline, on inspecte un echantillon, puis on serialise `dietetique.json` **en UTF-8 explicite**.
L'ancien cache du fork souffrait d'un defaut d'encodage (`Magn�esium`) : on **corrige la cause** (lecture
UTF-8 + ecriture UTF-8 sans BOM) plutot que de rafistoler la sortie.


In [6]:
var diet = Dietetique.Load(ciqual, recettes);

Console.WriteLine($"Constituants         : {diet.Constituants.Count}");
Console.WriteLine($"Denrees matchees     : {diet.Denrees.Count}");
Console.WriteLine($"Plats (avec denrees) : {diet.Plats.Count}");
Console.WriteLine();
Console.WriteLine("Denrees matchees (8 premieres, energie kJ/100g = constituant 0) :");
foreach (var d in diet.Denrees.Take(8))
    Console.WriteLine($"  {d.Nom,-30} -> energie {d.Compositions[0],8:0.#} kJ");
Console.WriteLine();
Console.WriteLine("Plats (5 premiers : creneau ordre_plat / nom / nb ingredients) :");
foreach (var p in diet.Plats.Take(5))
    Console.WriteLine($"  ordre {p.Ordre} | {p.Nom,-38} | {p.Ingredients.Count} ingredient(s)");

// Verification anti-mojibake : aucun caractere de remplacement U+FFFD.
int mojibake = diet.Constituants.Count(c => c.Nom.Contains('�'))
             + diet.Denrees.Count(d => d.Nom.Contains('�'))
             + diet.Plats.Count(p => p.Nom.Contains('�'));
Console.WriteLine();
Console.WriteLine($"Noms contenant U+FFFD (mojibake) : {mojibake}");

// Cache UTF-8 (artefact runtime, .gitignore).
var savePath = Path.Combine(DataDir, "dietetique.json");
File.WriteAllText(savePath, JsonConvert.SerializeObject(diet, Formatting.Indented), new UTF8Encoding(false));
Console.WriteLine($"Cache regenere : {Path.GetFileName(savePath)} ({new FileInfo(savePath).Length / 1024} Ko)");


Constituants         : 74


Denrees matchees     : 198


Plats (avec denrees) : 138


Denrees matchees (8 premieres, energie kJ/100g = constituant 0) :


  POIVRE GRIS MOULU K            -> energie     1380 kJ


...

  ordre 3 | RIZ CAMARGUAIS                         | 8 ingredient(s)


  ordre 1 | SAL MELANGE TENDRE                     | 2 ingredient(s)


  ordre 5 | ANANAS SIROP MORCEAUX                  | 2 ingredient(s)


Noms contenant U+FFFD (mojibake) : 0


Cache regenere : dietetique.json (448 Ko)


## 6. Exercices

Trois exercices autour de la **qualite de la fusion** (pas de Z3 ici — il arrive aux Slices C/D).
Completez les cellules : elles s'executent telles quelles (renvoient `null` / affichent un rappel) tant
qu'elles ne sont pas remplies.

### Exercice 1 — un matcheur alternatif (similarite de Jaccard)

`CalculeProximiteLexicale` est positionnel. Implementez une mesure **ensembliste** : la similarite de
Jaccard entre les deux sacs de mots, `|bag1 ∩ bag2| / |bag1 ∪ bag2|`, puis comparez l'aliment qu'elle
choisit pour quelques denrees a celui du matcheur d'origine.

- *Indice* : reutilisez `CreateSimpleList` pour produire les sacs, puis `Distinct`/`Intersect`/`Union`.
- *Etape 1* : ecrire `ProximiteJaccard(s1, s2)`.
- *Etape 2* : pour 3 denrees, afficher le meilleur aliment selon chaque mesure.


In [7]:
// Exercice 1 : similarite de Jaccard sur les sacs de mots.
static double ProximiteJaccard(string s1, string s2)
{
    // TODO etudiant : |intersection| / |union| des sacs de mots normalises.
    // Indice : var b1 = CreateSimpleList(s1).Distinct().ToList(); ...
    Console.WriteLine("Exercice 1 a completer");
    return 0.0; // valeur neutre tant que non implemente
}

// TODO etudiant : comparer, pour 3 denrees, l'aliment choisi par chaque mesure.


### Exercice 2 — auditer la qualite des appariements

Tous les matches ne se valent pas : certaines denrees n'ont aucun mot commun avec Ciqual et recoivent un
score faible (voire negatif a cause du malus). Calculez, pour chaque denree, sa **proximite maximale**, puis
listez les **10 pires** appariements — ceux qu'un humain devrait revoir.

- *Indice* : refaites le `Select` de `matchingMots` mais conservez le score, triez `OrderBy(Proximite)`.
- *Etape 1* : construire la liste `(denree, meilleurAliment, score)`.
- *Etape 2* : afficher les 10 plus faibles et commenter ce que le matcheur rate.


In [8]:
// Exercice 2 : top 10 des appariements les plus faibles.
List<(string Denree, string Aliment, double Score)> PiresAppariements(int n)
{
    // TODO etudiant : reconstruire (denree -> meilleur aliment + score), trier croissant, prendre n.
    Console.WriteLine("Exercice 2 a completer");
    return null;
}
// PiresAppariements(10) ...


### Exercice 3 — restrictions patient (`Min`/`Max` par constituant)

Le Demo2 contraint les menus par des **restrictions patient** (`Min`/`Max` par constituant) — c'est la
matiere de la Slice C cote solveur, mais la **structure de donnees** se construit deja ici. Ecrivez une
fonction qui, etant donne un nom de constituant et une borne max, renvoie la liste des denrees **au-dessus**
de cette borne (celles qu'un regime devrait limiter).

- *Indice* : retrouvez l'index du constituant via `diet.Constituants.FindIndex(c => c.Nom.Contains(...))`,
  puis filtrez `diet.Denrees` sur `d.Compositions[idx] > max`.
- *Etape 1* : `DenreesAuDessus(string constituant, decimal max)`.
- *Etape 2* : testez avec un constituant lipidique ou sodique.


In [9]:
// Exercice 3 : denrees depassant une borne sur un constituant donne.
List<string> DenreesAuDessus(string constituantContient, decimal max)
{
    // TODO etudiant : index du constituant, puis filtre sur diet.Denrees[].Compositions[idx] > max.
    Console.WriteLine("Exercice 3 a completer");
    return null;
}
// DenreesAuDessus("Sodium", 1000m) ...


## Conclusion — ce que Slice A etablit

Ce notebook **porte fidelement** `Dietetique.Load` : les deux jointures `CONST ⋈ COMPO ⋈ ALIM`, le matcheur
lexical flou `CalculeProximiteLexicale`, et l'agregation `Zip` des compositions de plats, sur les **vraies**
donnees Ciqual ANSES 2025 (3484 aliments, 258 000 compositions) et Recettes OpenData. Resultat : un catalogue
`Dietetique` de **74 constituants, ~200 denrees, 138 plats**, serialise en `dietetique.json` propre (UTF-8,
zero mojibake).

C'est la **couche de donnees** que les slices suivantes du capstone consommeront :

- **Slice B** — passer le corpus RecipeML a l'echelle (subsets cures -> montee en gamme).
- **Slice C** — le theoreme hierarchique `Patient -> Menus[7] -> Plats[5] -> Denree` + restrictions patient
  (le `PlanificateurDeMenus.Create` de la source, deja ebauche en exercice 3).
- **Slice D** — la **convergence a l'echelle** : garder Z3 tractable quand le pool de plats grandit.

La lecon transverse : LINQ absorbe **declarativement** une fusion de donnees reelle (cles absentes,
appariement flou, agregation vectorielle) la ou un script imperatif deviendrait un labyrinthe de boucles.
C'est le meme changement de paradigme — decrire *quoi*, pas *comment* — que la serie applique ensuite au
solveur.
